In [ ]:
# Import necessary libraries
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.utils.data.sampler import SubsetRandomSampler
from torchinfo import summary
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torchvision.utils import make_grid
from torchvision.models import vgg16, resnet50, mobilenet_v2

from PIL import Image, UnidentifiedImageError

from sklearn.model_selection import train_test_split

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [9]:
df_dir = '/home/ubuntu/visao-computacional/PetImages'

cat_files = os.listdir(os.path.join(df_dir, 'Cat'))
dog_files = os.listdir(os.path.join(df_dir, 'Dog'))


In [10]:
# limpar algumas imagens corrompidas do df
for folder in ['Cat', 'Dog']:
    folder_path = os.path.join(df_dir, folder)
    for filename in os.listdir(folder_path):
        file_path = os.path.join(folder_path, filename)
        try:
            with Image.open(file_path) as img:
                img.verify()
        except (IOError, SyntaxError, UnidentifiedImageError, ValueError):
            print(f"Removendo imagem corrompida: {file_path}")
            os.remove(file_path)

/home/ubuntu/visao-computacional/.venv/lib/python3.12/site-packages/PIL/TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))


In [ ]:
# Augmentation do dataframe, aplicamos diversas transformações em 100% do df.
# buscamos diversificar os exemplos que damos para o modelo, criando "réplicas" modificadas das imagens originais 
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(30),
    transforms.RandomRotation(30),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.2),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
    transforms.RandomApply([transforms.GaussianBlur(kernel_size=3)], p=0.5)
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# split dos dados, 0.7 test e 0.3 val
full_dataset = ImageFolder(df_dir, transform=None)
train_size = int(0.7 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_indices, val_indices = torch.utils.data.random_split(
    range(len(full_dataset)), [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
    )

In [14]:
def safe_loader(path):
    try:
        return Image.open(path).convert('RGB')
    except (UnidentifiedImageError, OSError, ValueError) as e:
        print(f"Erro ao abrir {path}: {e}")
        # Retorna uma imagem preta se der erro
        return Image.fromarray(np.zeros((224, 224, 3), dtype=np.uint8))
    
full_dataset = ImageFolder(df_dir, loader=safe_loader, transform=None)
train_dataset = ImageFolder(df_dir, loader=safe_loader, transform=train_transform)
val_dataset = ImageFolder(df_dir, loader=safe_loader, transform=val_transform)

train_sampler = SubsetRandomSampler(train_indices.indices)
val_sampler = SubsetRandomSampler(val_indices.indices)

train_loader = DataLoader(
    train_dataset, batch_size=32, sampler=train_sampler, num_workers=8
)
val_loader = DataLoader(
    val_dataset, batch_size=32, sampler=val_sampler, num_workers=8
)

print(f"Classes: {train_dataset.classes}")
print(f"Total de amostras: {len(full_dataset)}")
print(f"Amostras de treino: {len(train_indices)}")
print(f"Amostras de validação: {len(val_indices)}")

Classes: ['Cat', 'Dog']
Total de amostras: 24998
Amostras de treino: 17498
Amostras de validação: 7500


In [35]:
# --- Transfer Learning: VGG16 ---
model = vgg16(weights='IMAGENET1K_V1')
model.classifier[6] = nn.Linear(4096, 2)
model = model.to(device)

optimizer = optim.Adam(model.parameters(), lr=0.0001)
criterion = nn.CrossEntropyLoss()
num_epochs = 10
best_acc = 0.0

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    for inputs, labels in tqdm(train_loader, desc=f"VGG16 Epoch {epoch+1}/{num_epochs}"):
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    epoch_loss = running_loss / total
    epoch_acc = correct / total
    print(f"VGG16 Época {epoch+1}: Loss treino={epoch_loss:.4f} | Acc treino={epoch_acc:.4f}")

    # Validação
    model.eval()
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()
    val_acc = val_correct / val_total
    print(f"VGG16 Validação: Acc={val_acc:.4f}")

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), 'modelo_catsdogs_vgg16.pth')
        print("Novo melhor modelo VGG16 salvo!")

VGG16 Epoch 1/10:  15%|█▌        | 84/547 [00:39<03:31,  2.19it/s]/home/ubuntu/visao-computacional/.venv/lib/python3.12/site-packages/PIL/TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))
VGG16 Epoch 1/10: 100%|██████████| 547/547 [04:10<00:00,  2.18it/s]

VGG16 Época 1: Loss treino=0.1254 | Acc treino=0.9493


VGG16 Validação: Acc=0.9796
Novo melhor modelo VGG16 salvo!


VGG16 Epoch 2/10:  90%|█████████ | 494/547 [03:45<00:24,  2.20it/s]/home/ubuntu/visao-computacional/.venv/lib/python3.12/site-packages/PIL/TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))
VGG16 Epoch 2/10: 100%|██████████| 547/547 [04:09<00:00,  2.19it/s]

VGG16 Época 2: Loss treino=0.0755 | Acc treino=0.9710


...

VGG16 Época 9: Loss treino=0.0340 | Acc treino=0.9870


VGG16 Validação: Acc=0.9825


VGG16 Epoch 10/10:  29%|██▉       | 158/547 [01:13<02:56,  2.20it/s]/home/ubuntu/visao-computacional/.venv/lib/python3.12/site-packages/PIL/TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))
VGG16 Epoch 10/10: 100%|██████████| 547/547 [04:09<00:00,  2.19it/s]

VGG16 Época 10: Loss treino=0.0352 | Acc treino=0.9875


VGG16 Validação: Acc=0.9829


In [ ]:
# --- Transfer Learning: ResNet50 ---
model = resnet50(weights='IMAGENET1K_V1')
model.fc = nn.Linear(model.fc.in_features, 2)
model = model.to(device)

optimizer = optim.Adam(model.parameters(), lr=0.0001)
criterion = nn.CrossEntropyLoss()
num_epochs = 10
best_acc = 0.0

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    for inputs, labels in tqdm(train_loader, desc=f"ResNet50 Epoch {epoch+1}/{num_epochs}"):
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    epoch_loss = running_loss / total
    epoch_acc = correct / total
    print(f"ResNet50 Época {epoch+1}: Loss treino={epoch_loss:.4f} | Acc treino={epoch_acc:.4f}")

    # Validação
    model.eval()
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()
    val_acc = val_correct / val_total
    print(f"ResNet50 Validação: Acc={val_acc:.4f}")

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), 'modelo_catsdogs_resnet50.pth')
        print("Novo melhor modelo ResNet50 salvo!")

Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /home/ubuntu/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 220MB/s]
ResNet50 Epoch 1/10:  61%|██████    | 331/547 [01:43<01:03,  3.40it/s]/home/ubuntu/visao-computacional/.venv/lib/python3.12/site-packages/PIL/TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))
ResNet50 Epoch 1/10: 100%|██████████| 547/547 [02:48<00:00,  3.25it/s]

ResNet50 Época 1: Loss treino=0.0865 | Acc treino=0.9659


ResNet50 Validação: Acc=0.9876
Novo melhor modelo ResNet50 salvo!


ResNet50 Epoch 2/10:  58%|█████▊    | 315/547 [01:34<01:08,  3.39it/s]/home/ubuntu/visao-computacional/.venv/lib/python3.12/site-packages/PIL/TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))
ResNet50 Epoch 2/10: 100%|██████████| 547/547 [02:43<00:00,  3.35it/s]

...

ResNet50 Época 9: Loss treino=0.0242 | Acc treino=0.9917


ResNet50 Validação: Acc=0.9801


ResNet50 Epoch 10/10:  32%|███▏      | 177/547 [00:53<01:49,  3.38it/s]/home/ubuntu/visao-computacional/.venv/lib/python3.12/site-packages/PIL/TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))
ResNet50 Epoch 10/10: 100%|██████████| 547/547 [02:43<00:00,  3.35it/s]

ResNet50 Época 10: Loss treino=0.0258 | Acc treino=0.9910


ResNet50 Validação: Acc=0.9880


In [ ]:
# --- Transfer Learning: MobileNetV2 ---
model = mobilenet_v2(weights='IMAGENET1K_V1')
model.classifier[1] = nn.Linear(model.last_channel, 2)
model = model.to(device)

optimizer = optim.Adam(model.parameters(), lr=0.0001)
criterion = nn.CrossEntropyLoss()
num_epochs = 10
best_acc = 0.0

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    for inputs, labels in tqdm(train_loader, desc=f"MobileNetV2 Epoch {epoch+1}/{num_epochs}"):
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    epoch_loss = running_loss / total
    epoch_acc = correct / total
    print(f"MobileNetV2 Época {epoch+1}: Loss treino={epoch_loss:.4f} | Acc treino={epoch_acc:.4f}")

    # Validação
    model.eval()
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()
    val_acc = val_correct / val_total
    print(f"MobileNetV2 Validação: Acc={val_acc:.4f}")

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), 'modelo_catsdogs_mobilenetv2.pth')
        print("Novo melhor modelo MobileNetV2 salvo!")

Downloading: "https://download.pytorch.org/models/mobilenet_v2-b0353104.pth" to /home/ubuntu/.cache/torch/hub/checkpoints/mobilenet_v2-b0353104.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 222MB/s]
MobileNetV2 Epoch 1/10:  28%|██▊       | 151/547 [00:22<00:54,  7.26it/s]/home/ubuntu/visao-computacional/.venv/lib/python3.12/site-packages/PIL/TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))
MobileNetV2 Epoch 1/10: 100%|██████████| 547/547 [01:17<00:00,  7.06it/s]

MobileNetV2 Época 1: Loss treino=0.1091 | Acc treino=0.9553


MobileNetV2 Validação: Acc=0.9872
Novo melhor modelo MobileNetV2 salvo!


MobileNetV2 Epoch 2/10:  19%|█▉        | 106/547 [00:16<01:01,  7.12it/s]/home/ubuntu/visao-computacional/.venv/lib/python3.12/site-packages/PIL/TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))
MobileNetV2 Epoch 2/10: 100%|██████████| 547/547 [01:16<00:00,  7.12it/s]

...

MobileNetV2 Época 9: Loss treino=0.0228 | Acc treino=0.9921


MobileNetV2 Validação: Acc=0.9859


MobileNetV2 Epoch 10/10:  17%|█▋        | 95/547 [00:15<01:03,  7.17it/s]/home/ubuntu/visao-computacional/.venv/lib/python3.12/site-packages/PIL/TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))
MobileNetV2 Epoch 10/10: 100%|██████████| 547/547 [01:17<00:00,  7.05it/s]

MobileNetV2 Época 10: Loss treino=0.0215 | Acc treino=0.9913


MobileNetV2 Validação: Acc=0.9868
